In [1]:
from __future__ import annotations

import os
import sys
from datetime import datetime, timedelta, timezone
from decimal import Decimal
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not find project root containing pyproject.toml and src/")


PROJECT_ROOT = find_project_root(Path.cwd())

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

load_dotenv(PROJECT_ROOT / ".env")

print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {sys.version.split()[0]}")

Project root: d:\data\development\crypto
Python: 3.14.0


## Configuration Check

In [2]:
def mask(value: str | None, visible: int = 4) -> str:
    if not value:
        return "<missing>"
    if len(value) <= visible * 2:
        return "*" * len(value)
    return f"{value[:visible]}...{value[-visible:]}"


env_snapshot = {
    "COINBASE_API_KEY": mask(os.getenv("COINBASE_API_KEY")),
    "COINBASE_API_SECRET": mask(os.getenv("COINBASE_API_SECRET")),
    "DATABASE_URL": "<set>" if os.getenv("DATABASE_URL") else "<missing>",
    "COINBASE_BASE_URL": os.getenv("COINBASE_BASE_URL", "https://api.coinbase.com"),
}

pd.DataFrame(env_snapshot.items(), columns=["setting", "value"])

,setting,value
0,COINBASE_API_KEY,orga...c63a
1,COINBASE_API_SECRET,HK1G...uA==
2,DATABASE_URL,<set>
3,COINBASE_BASE_URL,https://api.coinbase.com


## Public Coinbase Connectivity

Coinbase Advanced Trade exposes public market endpoints under `/api/v3/brokerage/market/...`. These do not need credentials and are useful for separating basic connectivity from authenticated-client issues.

In [3]:
BASE_URL = os.getenv("COINBASE_BASE_URL", "https://api.coinbase.com").rstrip("/")


def public_get(path: str, params: dict | None = None) -> dict:
    url = f"{BASE_URL}{path}"
    response = requests.get(
        url,
        params=params,
        headers={"Accept": "application/json", "Cache-Control": "no-cache"},
        timeout=30,
    )
    print(f"GET {response.url} -> {response.status_code}")
    response.raise_for_status()
    return response.json()


server_time = public_get("/api/v3/brokerage/time")
server_time

GET https://api.coinbase.com/api/v3/brokerage/time -> 200


{'iso': '2026-08-07T01:41:46Z',
 'epochSeconds': '1786066906',
 'epochMillis': '1786066906674'}

In [4]:
public_products_response = public_get("/api/v3/brokerage/market/products")
public_products = public_products_response.get("products", [])

products_df = pd.DataFrame(public_products)
print(f"Products returned: {len(products_df):,}")

display_columns = [
    "product_id",
    "base_currency_id",
    "quote_currency_id",
    "price",
    "price_percentage_change_24h",
    "volume_24h",
    "status",
    "trading_disabled",
]
available_columns = [column for column in display_columns if column in products_df.columns]
products_df[available_columns].head(20)

GET https://api.coinbase.com/api/v3/brokerage/market/products -> 200
Products returned: 930


,product_id,base_currency_id,quote_currency_id,price,price_percentage_change_24h,volume_24h,status,trading_disabled
0,BTC-USD,BTC,USD,64292.09,-0.33510437268034,4454.76949207,online,False
1,BTC-USDC,BTC,USDC,64292.09,-0.33510437268034,4454.76949207,online,False
2,ETH-USD,ETH,USD,1899.34,-0.28455028455028,74102.08153124,online,False
3,ETH-USDC,ETH,USDC,1899.34,-0.28455028455028,74102.08153124,online,False
4,XRP-USD,XRP,USD,1.0354,-2.08983451536643,53200545.90925,online,False
5,XRP-USDC,XRP,USDC,1.0354,-2.08983451536643,53200545.90925,online,False
6,ADA-USD,ADA,USD,0.20034,5.02752293577982,228723879.72633432,online,False
7,ADA-USDC,ADA,USDC,0.20034,5.02752293577982,228723879.72633432,online,False
8,USDT-USD,USDT,USD,0.99908,0.01101134168193,44265985.39,online,False
9,SOL-USD,SOL,USD,72.6,-1.57266811279826,456957.02483503,online,False


In [5]:
if "quote_currency_id" not in products_df.columns:
    raise KeyError("Coinbase products response did not include quote_currency_id")

usd_products_df = products_df.loc[products_df["quote_currency_id"].eq("USD")].copy()

for numeric_column in ["price", "volume_24h"]:
    if numeric_column in usd_products_df.columns:
        usd_products_df[numeric_column] = pd.to_numeric(usd_products_df[numeric_column], errors="coerce")

usd_products_df = usd_products_df.sort_values("volume_24h", ascending=False, na_position="last")
print(f"USD-quoted products: {len(usd_products_df):,}")

usd_products_df[available_columns].head(30)

USD-quoted products: 406


,product_id,base_currency_id,quote_currency_id,price,price_percentage_change_24h,volume_24h,status,trading_disabled
68,PEPE-USD,PEPE,USD,2.800000e-06,-2.4390243902439,8.315324e+11,online,False
151,SHIB-USD,SHIB,USD,4.670000e-06,-3.31262939958592,1.103577e+11,online,False
208,BONK-USD,BONK,USD,2.810000e-06,0.7168458781362,9.491094e+10,online,False
749,MOG-USD,MOG,USD,1.000000e-07,0,7.507112e+10,online,False
754,NEX-USD,NEX,USD,1.480000e-06,2.06896551724138,4.986191e+09,online,False
43,PUMP-USD,PUMP,USD,2.385000e-03,-0.91400083090985,1.957084e+09,online,False
166,BNKR-USD,BNKR,USD,3.123000e-04,-4.93150684931507,1.450588e+09,online,False
716,NOICE-USD,NOICE,USD,9.900000e-06,2.06185567010309,1.027047e+09,online,False
11,HFT-USD,HFT,USD,3.020000e-02,69.66292134831461,9.428268e+08,online,False
608,FLOKI-USD,FLOKI,USD,2.140000e-05,3.88349514563107,8.769084e+08,online,False


## Market Rates: Ticker and Candles

In [6]:
sample_product_id = "BTC-USD"

ticker_response = public_get(f"/api/v3/brokerage/market/products/{sample_product_id}/ticker", params={"limit": 20})
ticker_summary = {
    "product_id": sample_product_id,
    "best_bid": ticker_response.get("best_bid"),
    "best_ask": ticker_response.get("best_ask"),
    "trades_returned": len(ticker_response.get("trades", [])),
}

pd.DataFrame([ticker_summary])

GET https://api.coinbase.com/api/v3/brokerage/market/products/BTC-USD/ticker?limit=20 -> 200


,product_id,best_bid,best_ask,trades_returned
0,BTC-USD,64294.5,64294.51,20


In [7]:
end = datetime.now(timezone.utc)
start = end - timedelta(days=7)

public_candles_response = public_get(
    f"/api/v3/brokerage/market/products/{sample_product_id}/candles",
    params={
        "granularity": "ONE_DAY",
        "start": int(start.timestamp()),
        "end": int(end.timestamp()),
    },
)

candles_df = pd.DataFrame(public_candles_response.get("candles", []))
if not candles_df.empty:
    candles_df["start"] = pd.to_datetime(candles_df["start"].astype(int), unit="s", utc=True)
    for column in ["low", "high", "open", "close", "volume"]:
        candles_df[column] = pd.to_numeric(candles_df[column], errors="coerce")
    candles_df = candles_df.sort_values("start")

candles_df

GET https://api.coinbase.com/api/v3/brokerage/market/products/BTC-USD/candles?granularity=ONE_DAY&start=1785462109&end=1786066909 -> 200


,start,low,high,open,close,volume
6,2026-08-01 00:00:00+00:00,62209.81,63093.00,62826.69,62764.20,3060.554042
5,2026-08-02 00:00:00+00:00,62743.87,63730.49,62764.20,63499.49,3283.947240
4,2026-08-03 00:00:00+00:00,62210.14,64032.43,63499.49,63466.51,7086.703286
3,2026-08-04 00:00:00+00:00,63250.00,64497.24,63466.51,64050.51,6227.774363
2,2026-08-05 00:00:00+00:00,63814.80,64988.78,64050.51,64603.03,6932.344050
1,2026-08-06 00:00:00+00:00,64087.41,64944.16,64603.02,64267.30,4520.432062
0,2026-08-07 00:00:00+00:00,64170.41,64425.57,64267.30,64289.92,148.931250


## Authenticated Read-Only Client Check

This section uses the existing `cryptoquant.collectors.coinbase_client.CoinbaseClient`. If this fails while the public checks pass, the likely issue is credential format, permission, or request signing rather than Coinbase availability.

In [8]:
from cryptoquant.collectors.coinbase_client import CoinbaseAPIError, CoinbaseClient
from cryptoquant.collectors.models import CandleGranularity


client = CoinbaseClient()

try:
    authenticated_products = client.get_products(active_only=True)
    print(f"Authenticated products returned: {len(authenticated_products):,}")
    authenticated_products_df = pd.DataFrame([product.model_dump() for product in authenticated_products])
    display(authenticated_products_df.head(20))
except Exception as exc:
    print(type(exc).__name__)
    print(exc)
    authenticated_products_df = pd.DataFrame()

ModuleNotFoundError: No module named 'pydantic'

In [9]:
try:
    authenticated_candles = client.get_candles(
        sample_product_id,
        granularity=CandleGranularity.ONE_DAY,
        days=7,
    )
    authenticated_candles_df = pd.DataFrame([candle.model_dump() for candle in authenticated_candles])
    print(f"Authenticated candles returned: {len(authenticated_candles_df):,}")
    display(authenticated_candles_df)
except Exception as exc:
    print(type(exc).__name__)
    print(exc)
    authenticated_candles_df = pd.DataFrame()

NameError
name 'client' is not defined


## Optional Raw Authenticated Best Bid/Ask

This calls a read-only/view endpoint using the client's low-level request helper.

In [ ]:
try:
    best_bid_ask = client._make_request(
        "GET",
        "/api/v3/brokerage/best_bid_ask",
        params={"product_ids": sample_product_id},
    )
    pricebooks_df = pd.DataFrame(best_bid_ask.get("pricebooks", []))
    display(pricebooks_df)
except Exception as exc:
    print(type(exc).__name__)
    print(exc)